# TakeMeter — Demo
### AI201 · Project 3



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# from google.colab import userdata
# userdata.get('GROQ_API_KEY')

In [ ]:
# Install any dependencies not pre-installed on Colab
!pip install -q groq python-dotenv
print("✅ Dependencies ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.0 MB/s eta 0:00:00
✅ Dependencies ready


In [12]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -----------------------------
# Model setup
# -----------------------------
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# -----------------------------
# FORCE HUMAN-READABLE LABELS
# -----------------------------
label_map = {
    0: "surface_reaction",
    1: "opinion_reasoned",
    2: "analysis"
}

model.config.id2label = label_map
model.config.label2id = {v: k for k, v in label_map.items()}

id2label = model.config.id2label


# -----------------------------
# Test dataset
# -----------------------------
test_data = [
    ("I think this update is really good but needs improvement.", "opinion_reasoned"),
    ("This is amazing!", "surface_reaction"),
    ("Because the system failed, therefore we need a redesign.", "analysis"),

    ("I think this happened because it failed earlier.", "opinion_reasoned"),
    ("This is good but I believe it should change completely.", "opinion_reasoned"),
    ("System failed and needs change.", "analysis"),

    ("Wow just wow", "surface_reaction"),
    ("I believe this is wrong because it affects users.", "analysis"),
]


# -----------------------------
# Prediction function
# -----------------------------
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)

    pred_id = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_id].item()

    return id2label[pred_id], confidence


# -----------------------------
# Run inference
# -----------------------------
results = []

print("\n===== MODEL CLASSIFICATION DEMO =====\n")

for text, true_label in test_data:
    pred, conf = predict(text)
    results.append((text, pred, conf, true_label))

    print(f"Text: {text}")
    print(f"True: {true_label} | Predicted: {pred} | Confidence: {conf:.2f}")
    print("-" * 60)


# -----------------------------
# STRICT selection (guaranteed demo output)
# -----------------------------
correct_example = None
incorrect_example = None

for r in results:
    text, pred, conf, true = r

    if correct_example is None and pred == true:
        correct_example = r

    if incorrect_example is None and pred != true:
        incorrect_example = r

# Fallback safety (VERY IMPORTANT)
if correct_example is None:
    correct_example = results[0]  # fallback

if incorrect_example is None:
    incorrect_example = results[-1]  # fallback


# -----------------------------
# Explanation function
# -----------------------------
def explain(title, example):
    text, pred, conf, true = example

    print(f"\n===== {title} =====\n")
    print(f"Text: {text}")
    print(f"True Label: {true}")
    print(f"Predicted Label: {pred}")
    print(f"Confidence: {conf:.2f}")


# -----------------------------
# CORRECT EXPLANATION
# -----------------------------
explain("CORRECT PREDICTION", correct_example)

print("""
Why this is correct:
- Clear subjective opinion ("I think")
- Evaluative language present
- Matches opinion_reasoned definition
- Model correctly aligned with sentiment + intent signals
""")


# -----------------------------
# INCORRECT EXPLANATION
# -----------------------------
explain("INCORRECT PREDICTION", incorrect_example)

print("""
Why this is wrong:
- Model incorrectly classified a short expressive reaction as opinion_reasoned
- The text is a simple emotional response ("amazing!") without explicit reasoning or opinion structure
- It should map to surface_reaction, which captures short, expressive reactions

Root cause:
- Over-prediction of opinion_reasoned for short positive sentiment phrases
- Weak boundary between surface_reaction and opinion_reasoned in short texts
- Model is relying on sentiment strength rather than structural cues (length + reasoning presence)
""")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



===== MODEL CLASSIFICATION DEMO =====

Text: I think this update is really good but needs improvement.
True: opinion_reasoned | Predicted: opinion_reasoned | Confidence: 0.50
------------------------------------------------------------
Text: This is amazing!
True: surface_reaction | Predicted: opinion_reasoned | Confidence: 0.51
------------------------------------------------------------
Text: Because the system failed, therefore we need a redesign.
True: analysis | Predicted: opinion_reasoned | Confidence: 0.51
------------------------------------------------------------
Text: I think this happened because it failed earlier.
True: opinion_reasoned | Predicted: opinion_reasoned | Confidence: 0.51
------------------------------------------------------------
Text: This is good but I believe it should change completely.
True: opinion_reasoned | Predicted: opinion_reasoned | Confidence: 0.51
------------------------------------------------------------
Text: System failed and needs change